# Data Deduplication and Quality Filtering

**Key insight:** Data quality consistently matters more than model architecture at scale. Filtering out 90% of a dataset often *improves* model quality — more training steps on clean data beats more data points of mixed quality.

This notebook covers the full data quality pipeline:

1. **Perceptual hashing** — fast exact/near-exact duplicate detection (dHash, pHash)
2. **Embedding-based deduplication** — semantic duplicate detection via cosine similarity clustering
3. **Hierarchical quality filtering** — the "data pyramid" pattern: resolution → aspect ratio → sharpness → aesthetics → safety
4. **Video-specific quality** — motion score, temporal consistency, scene coherence
5. **End-to-end pipeline** — chaining everything into a `DataCurationPipeline` with a `CurationReport`

**Reference papers:**
- Open-Sora 2.0 data pipeline patterns (arXiv 2503.09642)
- HunyuanVideo data curation (arXiv 2412.03603)

**VRAM:** ~2–4 GB (embedding model pass) | **Time:** ~2–3 hours total for a real dataset

In [ ]:
import hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFilter
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
from dataclasses import dataclass, field


## Perceptual Hashing

Unlike cryptographic hashes (SHA256, MD5), perceptual hashes are designed so that **similar images produce similar hash values**. Small visual differences produce small Hamming distances between hashes.

**Two main approaches:**

- **dHash (difference hash):** Resize to `(hash_size+1, hash_size)`, compare each pixel to the one to its right. One bit per comparison → `hash_size²` bits total. Very fast, excellent for exact and near-exact duplicates (identical content, minor crop/resize).

- **pHash (perceptual hash, DCT-based):** Resize to `(highfreq_factor * hash_size)²`, apply 2D DCT, keep the top-left `hash_size × hash_size` low-frequency coefficients, threshold at the median. More robust to JPEG compression artifacts, color adjustments, and moderate resizing than dHash.

**Production usage:** Both are O(1) per image and O(N²) for pairwise comparison (or O(N log N) with LSH). Use them as the **first pass** before any expensive embedding-based dedup. Typical threshold: Hamming distance ≤ 5 for dHash, ≤ 10 for pHash (on 64-bit hashes).

In [ ]:
# EXERCISE: Implement perceptual hashing for image deduplication

def dhash(image: Image.Image, hash_size: int = 8) -> int:
    """Difference hash: compare adjacent pixels horizontally.

    Args:
        image: Input PIL image.
        hash_size: Grid size; produces hash_size**2 bits.

    Returns:
        Integer hash value.
    """
    # YOUR CODE: resize to grayscale, compare adjacent pixels, pack bits
    raise NotImplementedError


def phash(image: Image.Image, hash_size: int = 8, highfreq_factor: int = 4) -> int:
    """DCT-based perceptual hash. More robust to compression/scaling than dHash.

    Args:
        image: Input PIL image.
        hash_size: Output hash is hash_size**2 bits.
        highfreq_factor: Resize multiplier before DCT.

    Returns:
        Integer hash value.
    """
    # YOUR CODE: resize, compute 2D DCT, threshold low-frequency coefficients at median
    raise NotImplementedError


def hamming_distance(hash1: int, hash2: int) -> int:
    """Count differing bits between two integer hashes."""
    return bin(hash1 ^ hash2).count("1")


def find_duplicates_hash(
    images: list[Image.Image],
    threshold: int = 5,
    use_phash: bool = False,
) -> list[tuple[int, int]]:
    """Find duplicate pairs using perceptual hashing.

    Args:
        images: List of PIL images.
        threshold: Max Hamming distance to consider a duplicate.
        use_phash: Use pHash instead of dHash.

    Returns:
        List of (i, j) index pairs that are duplicates.
    """
    # YOUR CODE: hash all images, compare all pairs, return those within threshold
    raise NotImplementedError


# --- Test ---
rng = np.random.default_rng(42)

def make_test_images() -> tuple[list[Image.Image], list[str]]:
    """Synthetic test images: base + variants + unrelated."""
    base_arr = rng.integers(50, 200, (128, 128, 3), dtype=np.uint8)
    base = Image.fromarray(base_arr)

    # Variant 1: minor resize (near-exact duplicate)
    resized = base.resize((120, 120)).resize((128, 128))
    # Variant 2: slight rotation
    rotated = base.rotate(3, expand=False)
    # Variant 3: JPEG compression artifacts
    import io
    buf = io.BytesIO()
    base.save(buf, format="JPEG", quality=30)
    buf.seek(0)
    jpeg_compressed = Image.open(buf).copy()
    # Unrelated image
    unrelated_arr = rng.integers(0, 255, (128, 128, 3), dtype=np.uint8)
    unrelated = Image.fromarray(unrelated_arr)

    images = [base, resized, rotated, jpeg_compressed, unrelated]
    labels = ["base", "resized", "rotated", "jpeg_q30", "unrelated"]
    return images, labels


test_imgs, test_labels = make_test_images()

dhashes = [dhash(img) for img in test_imgs]
phashes = [phash(img) for img in test_imgs]

print("=== Hamming Distances (dHash) ===")
print(f"{'':12s}", end="")
for lbl in test_labels:
    print(f"{lbl:>12s}", end="")
print()
for i, lbl_i in enumerate(test_labels):
    print(f"{lbl_i:12s}", end="")
    for j, _ in enumerate(test_labels):
        dist = hamming_distance(dhashes[i], dhashes[j])
        print(f"{dist:>12d}", end="")
    print()

print("\n=== Duplicate Pairs Found (dHash, threshold=10) ===")
dup_pairs = find_duplicates_hash(test_imgs, threshold=10, use_phash=False)
for i, j in dup_pairs:
    dist = hamming_distance(dhashes[i], dhashes[j])
    print(f"  ({test_labels[i]}, {test_labels[j]}) -> Hamming dist = {dist}")
if not dup_pairs:
    print("  None found at threshold=10")

print("\n=== Duplicate Pairs Found (pHash, threshold=10) ===")
dup_pairs_p = find_duplicates_hash(test_imgs, threshold=10, use_phash=True)
for i, j in dup_pairs_p:
    dist = hamming_distance(phashes[i], phashes[j])
    print(f"  ({test_labels[i]}, {test_labels[j]}) -> Hamming dist = {dist}")
if not dup_pairs_p:
    print("  None found at threshold=10")

## Embedding-Based Deduplication

Perceptual hashing catches near-exact duplicates but misses **semantic duplicates**: same scene photographed from a slightly different angle, same frame extracted at t and t+0.1s, or same image with a different color grade applied. These are functionally redundant for training but have very different pixel values.

**Embedding-based dedup** maps images into a semantic vector space (e.g., via CLIP or DINOv2) and clusters by cosine similarity. Images in the same cluster are semantically interchangeable — keep one representative per cluster.

**Similarity matrix** for N images is O(N²) in memory. At N=100K images with d=512-dimensional embeddings:
- Embeddings: 100K × 512 × 4 bytes ≈ 200 MB (fine)
- Full similarity matrix: 100K × 100K × 4 bytes ≈ 40 GB (not fine)

**Production pattern at scale:**
1. Hash-based first pass → removes exact dupes, O(N)
2. Embedding-based second pass with **approximate nearest neighbors** (FAISS `IndexFlatIP`, ScaNN, or HNSW) → O(N log N), no full matrix needed
3. Connected components / union-find to build clusters → keep one per cluster

Below we implement the exact O(N²) version (correct for small datasets) and note where to swap in ANN at scale.

In [ ]:
class MockVisionEncoder(nn.Module):
    """Random projection encoder for testing. Replace with CLIP/DINOv2 in production."""

    def __init__(self, in_dim: int = 3 * 64 * 64, out_dim: int = 128, seed: int = 0):
        super().__init__()
        torch.manual_seed(seed)
        self.proj = nn.Linear(in_dim, out_dim, bias=False)
        nn.init.orthogonal_(self.proj.weight)
        for p in self.parameters():
            p.requires_grad_(False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, C, H, W) -> (B, out_dim)."""
        b = x.shape[0]
        return self.proj(x.view(b, -1))


class EmbeddingDeduplicator:
    """Deduplicate images by clustering in embedding space."""

    def __init__(self, threshold: float = 0.95):
        """Args:
            threshold: Cosine similarity above which two images are considered duplicates.
        """
        self.threshold = threshold

    def compute_embeddings(
        self,
        images: list[torch.Tensor],
        model: nn.Module,
        batch_size: int = 16,
    ) -> torch.Tensor:
        """Encode images in batches and L2-normalize.

        Args:
            images: List of (C, H, W) tensors.
            model: Vision encoder.
            batch_size: Inference batch size.

        Returns:
            (N, D) normalized embedding matrix.
        """
        model.eval()
        all_embs = []
        with torch.no_grad():
            for start in range(0, len(images), batch_size):
                batch = torch.stack(images[start : start + batch_size])
                emb = model(batch)
                emb = F.normalize(emb, dim=-1)
                all_embs.append(emb)
        return torch.cat(all_embs, dim=0)

    def cosine_similarity_matrix(self, embeddings: torch.Tensor) -> torch.Tensor:
        """Compute full pairwise cosine similarity via matmul (embeddings already L2-normalized).

        Args:
            embeddings: (N, D) normalized tensor.

        Returns:
            (N, N) similarity matrix in [-1, 1].
        """
        return embeddings @ embeddings.T  # equivalent to cosine sim when L2-normalized

    def find_clusters(self, sim_matrix: torch.Tensor) -> list[set[int]]:
        """Union-Find connected components: two nodes are connected if sim > threshold.

        Args:
            sim_matrix: (N, N) cosine similarity matrix.

        Returns:
            List of sets; each set contains indices of one duplicate cluster.
        """
        N = sim_matrix.shape[0]
        parent = list(range(N))

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(x: int, y: int) -> None:
            rx, ry = find(x), find(y)
            if rx != ry:
                parent[rx] = ry

        sim_np = sim_matrix.cpu().numpy()
        for i in range(N):
            for j in range(i + 1, N):
                if sim_np[i, j] > self.threshold:
                    union(i, j)

        cluster_map: dict[int, set[int]] = defaultdict(set)
        for i in range(N):
            cluster_map[find(i)].add(i)
        return list(cluster_map.values())

    def deduplicate(
        self,
        images: list[torch.Tensor],
        model: nn.Module,
    ) -> list[int]:
        """Return indices of images to keep (one per duplicate cluster).

        At scale: swap compute_embeddings + find_clusters for FAISS ANN search.
        """
        embeddings = self.compute_embeddings(images, model)
        sim_matrix = self.cosine_similarity_matrix(embeddings)
        clusters = self.find_clusters(sim_matrix)
        # Keep the first (lowest-index) element of each cluster
        keep = sorted(min(cluster) for cluster in clusters)
        return keep, embeddings, sim_matrix, clusters


# --- Build synthetic test data with known duplicate groups ---
torch.manual_seed(7)
N_TOTAL = 20
D_INPUT = 3 * 64 * 64  # flattened 64x64 RGB

# Groups: 0-4 are duplicates, 5-9 are duplicates, 10-14 are duplicates, 15-19 are unique
group_protos = torch.randn(7, D_INPUT)  # 3 groups + 5 unique
noise_scale = 0.01

synthetic_images: list[torch.Tensor] = []
true_groups: list[int] = []
img_labels: list[str] = []

for g in range(3):  # 3 duplicate groups of 4 each
    proto = group_protos[g]
    for k in range(4):
        noisy = proto + noise_scale * torch.randn_like(proto)
        # Reshape to (3, 64, 64) for the mock encoder
        synthetic_images.append(noisy.view(3, 64, 64))
        true_groups.append(g)
        img_labels.append(f"G{g}-{k}")

for u in range(8):  # 8 unique images
    unique = group_protos[3 + u % 4] + 5.0 * torch.randn(D_INPUT)  # far from groups
    synthetic_images.append(unique.view(3, 64, 64))
    true_groups.append(3 + u)
    img_labels.append(f"U{u}")

print(f"Total synthetic images: {len(synthetic_images)} ({3} duplicate groups of 4, 8 unique)")

model = MockVisionEncoder(in_dim=D_INPUT, out_dim=128)
deduplicator = EmbeddingDeduplicator(threshold=0.98)
keep_indices, embeddings, sim_matrix, clusters = deduplicator.deduplicate(synthetic_images, model)

dup_clusters = [c for c in clusters if len(c) > 1]
print(f"Found {len(dup_clusters)} duplicate groups")
print(f"Keeping {len(keep_indices)} of {len(synthetic_images)} images")
for clust in sorted(dup_clusters, key=min):
    names = [img_labels[i] for i in sorted(clust)]
    print(f"  Cluster: {names} -> keep idx {min(clust)} ({img_labels[min(clust)]})")

# --- Visualize similarity matrix ---
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_matrix.numpy(), vmin=0.0, vmax=1.0, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(img_labels)))
ax.set_yticks(range(len(img_labels)))
ax.set_xticklabels(img_labels, rotation=90, fontsize=8)
ax.set_yticklabels(img_labels, fontsize=8)
plt.colorbar(im, ax=ax, label="Cosine Similarity")
ax.set_title("Pairwise Cosine Similarity Matrix\n(bright squares = duplicate clusters)")
plt.tight_layout()
plt.show()


## Hierarchical Quality Filtering (Data Pyramid)

Raw scraped data is mostly garbage. The **data pyramid** pattern applies filters from cheapest to most expensive, reducing volume at each stage so that expensive filters only run on data that already passed cheap filters.

```
Raw data (100%)            ←  everything scraped
    │
    ▼ Resolution/format filter   (O(1), removes ~30–50%)
    │
    ▼ Aspect ratio filter        (O(1), removes ~5–10%)
    │
    ▼ Blur/sharpness filter      (O(pixels), removes ~10–20%)
    │
    ▼ Aesthetic scoring          (O(model inference), removes ~20–40%)
    │
    ▼ Safety/NSFW filter         (O(model inference), removes ~1–5%)
    │
Training data (10–30%)     ←  what you actually train on
```

**Why order by cost?** If your aesthetic model costs 10ms/image and you have 100M images:
- Running aesthetic first: 100M × 10ms = 1M seconds ≈ 11.6 days
- Running resolution first (removes 40%): 60M × 10ms = 600K seconds ≈ 7 days
- Both cheap filters first (removes 55%): 45M × 10ms = 450K seconds ≈ 5.2 days

The ordering saves wall-clock time proportional to how much cheap filters remove.

**Key design principle:** every filter returns `(pass: bool, score: float)` — the score enables soft filtering (rank by score, keep top K%) instead of hard binary cuts.

In [ ]:
from PIL import ImageStat


def resolution_filter(
    img: Image.Image,
    min_size: int = 512,
) -> tuple[bool, float]:
    """Pass if both width and height >= min_size.

    Score = min(W, H) / min_size, clamped to [0, 1].
    """
    w, h = img.size
    short_side = min(w, h)
    score = min(short_side / min_size, 1.0)
    return short_side >= min_size, score


def aspect_ratio_filter(
    img: Image.Image,
    min_ratio: float = 0.4,
    max_ratio: float = 2.5,
) -> tuple[bool, float]:
    """Pass if W/H is within [min_ratio, max_ratio].

    Score: 1.0 if ratio is near 1.0 (square-ish), lower toward limits.
    """
    w, h = img.size
    ratio = w / h
    passed = min_ratio <= ratio <= max_ratio
    # Score: 1.0 at ratio=1.0, falls off toward limits
    if ratio >= 1.0:
        score = 1.0 - (ratio - 1.0) / max(max_ratio - 1.0, 1e-6)
    else:
        score = ratio / max(min_ratio, 1e-6) - 1.0 + 1.0  # symmetric
        score = 1.0 - (1.0 - ratio) / max(1.0 - min_ratio, 1e-6)
    return passed, max(0.0, min(score, 1.0))


def sharpness_filter(
    img: Image.Image,
    min_variance: float = 100.0,
) -> tuple[bool, float]:
    """Laplacian variance as a sharpness proxy. Low = blurry.

    Args:
        img: Input image.
        min_variance: Threshold; empirically ~100 separates blurry from sharp.

    Returns:
        (passed, score) where score = variance / min_variance clamped to [0, 1].
    """
    gray = img.convert("L")
    laplacian = gray.filter(ImageFilter.Kernel(
        size=3,
        kernel=[0, 1, 0, 1, -4, 1, 0, 1, 0],
        scale=1,
        offset=0,
    ))
    lap_arr = np.array(laplacian, dtype=np.float32)
    variance = float(np.var(lap_arr))
    score = min(variance / max(min_variance, 1e-6), 1.0)
    return variance >= min_variance, score


def _colorfulness(img: Image.Image) -> float:
    """Hasler & Susstrunk colorfulness metric."""
    arr = np.array(img.convert("RGB"), dtype=np.float32)
    R, G, B = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    rg = R - G
    yb = 0.5 * (R + G) - B
    return float(np.sqrt(np.std(rg) ** 2 + np.std(yb) ** 2) + 0.3 * np.sqrt(np.mean(rg) ** 2 + np.mean(yb) ** 2))


def _contrast(img: Image.Image) -> float:
    """RMS contrast on grayscale."""
    gray = np.array(img.convert("L"), dtype=np.float32) / 255.0
    return float(np.std(gray))


def _edge_density(img: Image.Image) -> float:
    """Fraction of pixels that are edges (Sobel-like)."""
    gray = img.convert("L")
    sx = gray.filter(ImageFilter.Kernel(3, [-1, 0, 1, -2, 0, 2, -1, 0, 1], scale=4))
    sy = gray.filter(ImageFilter.Kernel(3, [-1, -2, -1, 0, 0, 0, 1, 2, 1], scale=4))
    mag = np.sqrt(np.array(sx, dtype=np.float32) ** 2 + np.array(sy, dtype=np.float32) ** 2)
    return float(np.mean(mag > 20))


def aesthetic_score_proxy(img: Image.Image) -> tuple[bool, float]:
    """Lightweight aesthetic proxy: colorfulness + contrast + edge density.

    In production, replace with a trained aesthetic predictor (e.g., LAION aesthetic model).
    Score in [0, 1]. Threshold at 0.3 as a soft gate.
    """
    color = min(_colorfulness(img) / 100.0, 1.0)
    contrast = min(_contrast(img) / 0.3, 1.0)
    edges = min(_edge_density(img) / 0.2, 1.0)
    score = 0.4 * color + 0.3 * contrast + 0.3 * edges
    return score >= 0.3, float(score)


@dataclass
class FilterStageResult:
    name: str
    input_count: int
    passed_count: int
    scores: list[float] = field(default_factory=list)

    @property
    def retention_rate(self) -> float:
        return self.passed_count / max(self.input_count, 1)


class QualityPipeline:
    """Chain quality filters and track per-stage statistics."""

    FILTERS = [
        ("resolution", resolution_filter),
        ("aspect_ratio", aspect_ratio_filter),
        ("sharpness", sharpness_filter),
        ("aesthetic", aesthetic_score_proxy),
    ]

    def run(
        self,
        images: list[Image.Image],
    ) -> tuple[list[Image.Image], list[FilterStageResult]]:
        """Run all filters in order; return surviving images and per-stage stats."""
        current = images
        results: list[FilterStageResult] = []

        for name, fn in self.FILTERS:
            passed = []
            scores = []
            for img in current:
                ok, score = fn(img)
                scores.append(score)
                if ok:
                    passed.append(img)
            stage = FilterStageResult(
                name=name,
                input_count=len(current),
                passed_count=len(passed),
                scores=scores,
            )
            results.append(stage)
            current = passed

        return current, results


# --- Generate 50 synthetic images with varying quality ---
def make_synthetic_dataset(n: int = 50) -> list[Image.Image]:
    """Mix of high-res sharp, low-res, blurry, and extreme-AR images."""
    rng2 = np.random.default_rng(123)
    imgs = []
    for i in range(n):
        kind = i % 5
        if kind == 0:  # good: large, normal AR
            arr = rng2.integers(30, 220, (600, 800, 3), dtype=np.uint8)
            img = Image.fromarray(arr)
        elif kind == 1:  # bad: too small
            arr = rng2.integers(0, 255, (200, 200, 3), dtype=np.uint8)
            img = Image.fromarray(arr)
        elif kind == 2:  # bad: blurry (constant color region)
            arr = np.full((600, 600, 3), 128, dtype=np.uint8)
            arr += rng2.integers(0, 3, arr.shape, dtype=np.uint8)
            img = Image.fromarray(arr)
        elif kind == 3:  # bad: extreme aspect ratio
            arr = rng2.integers(0, 255, (600, 100, 3), dtype=np.uint8)
            img = Image.fromarray(arr)
        else:  # borderline: medium size
            arr = rng2.integers(20, 240, (520, 520, 3), dtype=np.uint8)
            img = Image.fromarray(arr)
    imgs.append(img)
    return imgs


dataset = make_synthetic_dataset(50)
pipeline = QualityPipeline()
survivors, stage_results = pipeline.run(dataset)

print(f"{'Stage':<16} {'Input':>7} {'Passed':>7} {'Retention':>10}")
print("-" * 44)
prev = len(dataset)
for sr in stage_results:
    print(f"{sr.name:<16} {sr.input_count:>7} {sr.passed_count:>7} {sr.retention_rate:>9.1%}")
print(f"{'FINAL':<16} {len(dataset):>7} {len(survivors):>7} {len(survivors)/len(dataset):>9.1%}")

# --- Funnel chart ---
counts = [len(dataset)] + [sr.passed_count for sr in stage_results]
labels = ["Raw"] + [sr.name for sr in stage_results]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels[::-1], counts[::-1], color=colors[:len(counts)][::-1])
for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{count} ({count/len(dataset):.0%})", va="center", fontsize=9)
ax.set_xlabel("Number of Images")
ax.set_title("Quality Filtering Funnel")
ax.set_xlim(0, len(dataset) * 1.25)
plt.tight_layout()
plt.show()


## Video-Specific Quality

Video adds a **temporal dimension** that images don't have. A video can be visually sharp frame-by-frame but still be low quality for training due to:

| Problem | Symptom | Metric |
|---|---|---|
| Static/boring | No motion | Motion score too low |
| Noise/corruption | Extreme random motion | Motion score too high |
| Flickering | Large adjacent-frame differences | Low temporal consistency |
| Jump cuts | Unrelated frames in same clip | High CLIP embedding variance |
| Encoding artifacts | Block noise across frames | Low sharpness × high motion |

**Key metrics:**

- **Motion score:** Average optical flow magnitude across the clip. Computed via `cv2.calcOpticalFlowFarneback` in production; we implement a simple frame-difference fallback.
- **Temporal consistency (SSIM proxy):** Mean structural similarity between consecutive frames. SSIM measures luminance, contrast, and structure jointly — more perceptually meaningful than MSE. We implement an MSE-based proxy here.
- **Scene coherence:** Variance of per-frame embeddings. High variance → frames from different scenes were spliced together.

**Sampling strategy:** Don't process every frame — sample N frames uniformly. For a 10s clip at 24fps (240 frames), sampling 16 frames is sufficient for motion and consistency estimates. This gives a ~15× speedup.

In [ ]:
def _frame_diff_flow(frame1: np.ndarray, frame2: np.ndarray) -> float:
    """Frame-difference proxy for optical flow magnitude.

    Returns mean absolute difference in grayscale — zero when frames are identical,
    high when frames change dramatically. Normalized to [0, 255] scale.
    """
    g1 = frame1.mean(axis=2) if frame1.ndim == 3 else frame1
    g2 = frame2.mean(axis=2) if frame2.ndim == 3 else frame2
    return float(np.mean(np.abs(g1.astype(np.float32) - g2.astype(np.float32))))


def motion_score(frames: list[np.ndarray]) -> float:
    """Average inter-frame motion magnitude across the clip.

    Args:
        frames: List of (H, W, 3) uint8 arrays in chronological order.

    Returns:
        Mean frame-difference magnitude; higher = more motion.

    Production note: Replace _frame_diff_flow with cv2.calcOpticalFlowFarneback
    for more accurate motion estimation.
    """
    if len(frames) < 2:
        return 0.0
    diffs = [_frame_diff_flow(frames[i], frames[i + 1]) for i in range(len(frames) - 1)]
    return float(np.mean(diffs))


def _ssim_proxy(frame1: np.ndarray, frame2: np.ndarray) -> float:
    """MSE-based structural similarity proxy in [0, 1].

    Full SSIM requires gaussian windowing; this proxy captures the same
    gross-level temporal consistency with much less code.
    Returns 1.0 for identical frames, ~0.0 for completely different frames.
    """
    f1 = frame1.astype(np.float32) / 255.0
    f2 = frame2.astype(np.float32) / 255.0
    mse = float(np.mean((f1 - f2) ** 2))
    # Convert MSE to similarity: 1 / (1 + k*MSE), k tuned so MSE=0.01 -> sim~0.9
    return 1.0 / (1.0 + 100.0 * mse)


def temporal_consistency(frames: list[np.ndarray]) -> float:
    """Mean pairwise SSIM-proxy between consecutive frames.

    Args:
        frames: List of (H, W, 3) uint8 arrays.

    Returns:
        Consistency score in [0, 1]; 1.0 = perfectly stable clip.
    """
    if len(frames) < 2:
        return 1.0
    scores = [_ssim_proxy(frames[i], frames[i + 1]) for i in range(len(frames) - 1)]
    return float(np.mean(scores))


@dataclass
class VideoQualityReport:
    motion: float
    consistency: float
    min_resolution: int
    passed: bool
    reason: str = ""


class VideoQualityScorer:
    """Combined video quality scorer.

    Args:
        min_motion: Frames must have some motion (avoid static clips).
        max_motion: Too much motion = noise/corruption.
        min_consistency: Minimum temporal stability.
        min_resolution: Minimum short-side pixel count.
    """

    def __init__(
        self,
        min_motion: float = 1.0,
        max_motion: float = 50.0,
        min_consistency: float = 0.7,
        min_resolution: int = 256,
    ):
        self.min_motion = min_motion
        self.max_motion = max_motion
        self.min_consistency = min_consistency
        self.min_resolution = min_resolution

    def score(self, frames: list[np.ndarray]) -> VideoQualityReport:
        """Evaluate a video clip represented as a list of frames."""
        if not frames:
            return VideoQualityReport(0, 0, 0, False, "empty")

        h, w = frames[0].shape[:2]
        min_res = min(h, w)
        mot = motion_score(frames)
        cons = temporal_consistency(frames)

        reasons = []
        if min_res < self.min_resolution:
            reasons.append(f"low_res({min_res}px)")
        if mot < self.min_motion:
            reasons.append(f"static(motion={mot:.1f})")
        if mot > self.max_motion:
            reasons.append(f"noisy(motion={mot:.1f})")
        if cons < self.min_consistency:
            reasons.append(f"inconsistent(ssim={cons:.2f})")

        passed = len(reasons) == 0
        return VideoQualityReport(
            motion=mot,
            consistency=cons,
            min_resolution=min_res,
            passed=passed,
            reason=", ".join(reasons) if reasons else "ok",
        )


# --- Synthetic video generation ---
def make_bouncing_ball(n_frames: int = 16, size: int = 320) -> list[np.ndarray]:
    """Bouncing ball: moderate motion, high temporal consistency."""
    frames = []
    for t in range(n_frames):
        arr = np.full((size, size, 3), 40, dtype=np.uint8)
        cx = int(size * 0.5 + (size * 0.3) * np.sin(2 * np.pi * t / n_frames))
        cy = int(size * 0.5 + (size * 0.3) * np.cos(2 * np.pi * t / n_frames))
        rr, cc = np.ogrid[:size, :size]
        mask = (rr - cy) ** 2 + (cc - cx) ** 2 <= 20 ** 2
        arr[mask] = [200, 100, 50]
        frames.append(arr)
    return frames


def make_static_video(n_frames: int = 16, size: int = 320) -> list[np.ndarray]:
    """Static scene: zero motion (should fail min_motion)."""
    base = np.random.default_rng(5).integers(50, 200, (size, size, 3), dtype=np.uint8)
    return [base.copy() for _ in range(n_frames)]


def make_noise_video(n_frames: int = 16, size: int = 320) -> list[np.ndarray]:
    """Pure random noise: extreme motion, low consistency."""
    rng3 = np.random.default_rng(99)
    return [rng3.integers(0, 255, (size, size, 3), dtype=np.uint8) for _ in range(n_frames)]


def make_jump_cut_video(n_frames: int = 16, size: int = 320) -> list[np.ndarray]:
    """First half: scene A. Second half: completely different scene B."""
    rng4 = np.random.default_rng(7)
    scene_a = np.full((size, size, 3), 30, dtype=np.uint8)
    scene_b = np.full((size, size, 3), 200, dtype=np.uint8)
    frames = []
    for i in range(n_frames):
        base = scene_a if i < n_frames // 2 else scene_b
        noise = rng4.integers(0, 5, (size, size, 3), dtype=np.uint8)
        frames.append(np.clip(base.astype(int) + noise, 0, 255).astype(np.uint8))
    return frames


scorer = VideoQualityScorer(
    min_motion=1.0,
    max_motion=50.0,
    min_consistency=0.7,
    min_resolution=256,
)

test_videos = [
    ("bouncing_ball", make_bouncing_ball()),
    ("static_scene", make_static_video()),
    ("noise", make_noise_video()),
    ("jump_cut", make_jump_cut_video()),
]

print(f"{'Name':<18} {'Motion':>8} {'Consist.':>10} {'Res':>6} {'Pass':>6}  Reason")
print("-" * 70)
for name, frames in test_videos:
    report = scorer.score(frames)
    flag = "PASS" if report.passed else "FAIL"
    print(f"{name:<18} {report.motion:>8.2f} {report.consistency:>10.3f} "
          f"{report.min_resolution:>6} {flag:>6}  {report.reason}")


## End-to-End Pipeline

Combine deduplication and quality filtering into a single `DataCurationPipeline`. In production this runs as a distributed job (Ray, Spark, or a simple multiprocessing pool) over a dataset stored in object storage.

The `CurationReport` dataclass captures the full audit trail: how many samples were removed at each stage and why. This is critical for reproducibility — you want to be able to re-run the exact same pipeline on new data and get the same filtering behavior.

In [ ]:
# EXERCISE: Build the full data curation pipeline

@dataclass
class CurationReport:
    total_input: int
    after_dedup: int
    after_quality: int
    final_count: int
    rejection_reasons: dict[str, int] = field(default_factory=dict)
    stage_counts: dict[str, int] = field(default_factory=dict)

    def print_summary(self) -> None:
        print("=" * 50)
        print("DATA CURATION REPORT")
        print("=" * 50)
        print(f"  Total input:       {self.total_input:>8}  (100.0%)")
        pct = lambda n: f"{n/max(self.total_input,1):.1%}"
        print(f"  After dedup:       {self.after_dedup:>8}  ({pct(self.after_dedup)})")
        print(f"  After quality:     {self.after_quality:>8}  ({pct(self.after_quality)})")
        print(f"  Final:             {self.final_count:>8}  ({pct(self.final_count)})")
        print()
        print("  Rejection breakdown:")
        for reason, count in sorted(self.rejection_reasons.items(), key=lambda x: -x[1]):
            print(f"    {reason:<20} {count:>6}  ({pct(count)})")


class DataCurationPipeline:
    """Full curation pipeline: dedup → quality filter → report.

    Uses find_duplicates_hash() from above and QualityPipeline.FILTERS.

    Args:
        dedup_threshold: Hash Hamming distance threshold (0 = exact only).
        quality_filters: Override the default filter list.
    """

    def __init__(
        self,
        dedup_threshold: int = 5,
        quality_filters: list | None = None,
    ):
        self.dedup_threshold = dedup_threshold
        self._filters = quality_filters or QualityPipeline.FILTERS

    def run(self, dataset: list[Image.Image]) -> CurationReport:
        """Execute the full curation pipeline.

        Args:
            dataset: Raw input images.

        Returns:
            CurationReport with counts at each stage and rejection reasons.
        """
        # YOUR CODE:
        # 1. Dedup stage: use find_duplicates_hash, remove higher-index duplicates
        # 2. Quality filter stage: run each filter, track rejections by reason
        # 3. Return a CurationReport with all stats
        raise NotImplementedError


# --- Run on synthetic dataset ---
full_dataset = make_synthetic_dataset(50)

# Add a few deliberate duplicates
import io as _io
_buf = _io.BytesIO()
full_dataset[0].save(_buf, format="PNG")
_buf.seek(0)
dup_copy = Image.open(_buf).copy()
full_dataset.extend([dup_copy, dup_copy, dup_copy])  # 3 exact duplicates of item 0

pipeline = DataCurationPipeline(dedup_threshold=0)
report = pipeline.run(full_dataset)
report.print_summary()

## Checkpoint

**Key takeaways from this notebook:**

- **Hash dedup is O(N)** per image and should always be the first pass. At 1M images it runs in seconds. Hamming distance threshold 0–5 for exact/near-exact, 5–15 for loose near-duplicates.

- **Embedding dedup catches semantic duplicates** (same scene, different angle/crop) that hashing misses. The similarity matrix is O(N²) — at N > 100K, switch to approximate nearest neighbors (FAISS `IndexFlatIP`, ScaNN, or HNSW). Union-find / connected components is the right data structure for cluster extraction.

- **Filter ordering matters**: cheap filters first, expensive last. A resolution filter that runs in microseconds should always precede an aesthetic model that runs in milliseconds. The cost savings compound multiplicatively.

- **The data pyramid**: start with 10× your target dataset size, expect to keep 10–30% after full curation. Filtering out 90% of data often *improves* downstream model quality — cleaner data means more efficient training signal per step.

- **Video quality adds temporal dimensions**: a blurry but temporally consistent video clip may be more valuable training signal than a sharp but flickering one. Motion scoring requires optical flow or frame differencing; temporal consistency uses SSIM or MSE between adjacent frames.

- **Audit everything**: `CurationReport` with per-stage counts and rejection reasons is non-negotiable. You will re-run the pipeline many times with different thresholds — traceability saves hours of debugging.

**Production reading:**
- Open-Sora 2.0 data pipeline: arXiv 2503.09642
- HunyuanVideo data curation: arXiv 2412.03603